<a href="https://colab.research.google.com/github/sanjeetworld/mini-llm-from-scratch/blob/main/Mini_LLM_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch datasets tqdm

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
texts = dataset["train"]["text"]

print(texts[:3])

['', ' = Valkyria Chronicles III = \n', '']


In [ ]:
chars = sorted(list(set("".join(texts))))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(text):
    return [stoi[c] for c in text if c in stoi]

def decode(tokens):
    return "".join([itos[t] for t in tokens])

print("Vocab size:", vocab_size)

Vocab size: 1013


In [ ]:
import torch

data = torch.tensor(encode(" ".join(texts)), dtype=torch.long)

block_size = 64

def get_batch(batch_size=32):
    ix = torch.randint(0, len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MiniLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 128)
        self.linear = nn.Linear(128, vocab_size)

    def forward(self, x, targets=None):
        x = self.embedding(x)
        logits = self.linear(x)

        if targets is None:
            return logits, None

        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            targets.view(-1)
        )
        return logits, loss

In [ ]:
model = MiniLLM()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(2000):
    x, y = get_batch()
    logits, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

Step 0 | Loss: 7.0306
Step 200 | Loss: 4.0223
Step 400 | Loss: 2.9121
Step 600 | Loss: 2.6841
Step 800 | Loss: 2.5591
Step 1000 | Loss: 2.5572
Step 1200 | Loss: 2.5039
Step 1400 | Loss: 2.4980
Step 1600 | Loss: 2.4801
Step 1800 | Loss: 2.5244


In [ ]:
def generate_text(start, max_len=100):
    model.eval()
    idx = torch.tensor([encode(start)], dtype=torch.long)

    for _ in range(max_len):
        logits, _ = model(idx)
        probs = torch.softmax(logits[:, -1, :], dim=-1)
        next_char = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_char], dim=1)

    return decode(idx[0].tolist())

print(generate_text("Artificial intelligence"))

Artificial intelligence = tutet . , Mang Gingenes  o ) = opom. . Alede .. l peck ved ogr .  t wef e t (in thed ediabee " re


In [ ]:
torch.save(model.state_dict(), "mini_llm.pth")